# Chapter 5 &mdash; Automd: a Markdown Language for All Machines

**Concept 12 of the Chapter 5 decomposition:** *Automd: a Markdown Language for All Machines*

Type a commented text file; `md2mc` turns it into a Jove machine and a drawing.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Automd-Markdown/Concept-Automd-Markdown.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Drawing machines by hand does not scale and does not diff. **Automd** is a small
markdown language: a header line naming the machine type, then one transition per
line, with `!!` comments to end of line.

`md2mc` parses it into the Jove dictionary; `dotObj_dfa` draws it. The same parser
handles `DFA`, `NFA`, `PDA` and `TM`, so one notation carries you through the whole
book.

Because the source is **text**, machines live in version control, get reviewed, and
carry comments explaining each transition.

## 2. Definitions

### From markdown, in one call

In [ ]:
src = '''DFA
!! Accepts strings over {0,1} with an even number of 0s.
IF : 0 -> Odd      !! one more 0 -> parity flips
IF : 1 -> IF       !! 1s do not matter
Odd: 0 -> IF
Odd: 1 -> Odd
'''
D = md2mc(src)

### and the machine you get back is an ordinary Python dict

In [ ]:
def show(D):
    for k in ['Q', 'Sigma', 'q0', 'F']:
        v = D[k]
        print("%-6s %s" % (k, sorted(v) if isinstance(v, set) else v))
    print("Delta:")
    for kv in sorted(D["Delta"].items()):
        print("   %s" % (kv,))

## 3. Tests

Parsed correctly, comments discarded.

In [ ]:
show(D)
assert D["q0"] == "IF" and D["F"] == {"IF"}

The markdown lives in a **text file**, so designs are diffable and reviewable.

In [ ]:
open('even0.dfa', 'w').write(src)
D2 = md2mc(open('even0.dfa').read())
print("round-tripped through a file, same machine? ", iso_dfa(D, D2))
assert iso_dfa(D, D2)
import os; os.remove('even0.dfa')

One notation, four machine types &mdash; the header line selects the parser.

In [ ]:
N = md2mc('''NFA
I  : 0 -> I
I  : 0 -> F
F  : '' -> I
''')
print("NFA states :", sorted(N["Q"]), "  (note the '' epsilon move)")
print("\nSame syntax shape for PDA and TM -- see Chapters 12 and 13.")

## 4. Animation

What `md2mc` built, drawn and animated.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(D, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Add a `!!` comment to every transition of a machine you wrote earlier.
2. What happens if you omit the `DFA` header line?
3. Keep a `.dfa` file in git and diff two versions. What does the diff show?

In [ ]:
# Your work for the exercises above.